# 03 · Análisis descriptivo y correlaciones

Estadísticas básicas y relaciones entre variables:

- Descriptivos de los sub-índices y del IRA.
- IRA por nivel, departamento y cultivo.
- Matriz de correlaciones de las 26 variables + IRA.
- Relación entre sub-índices y el score final.


In [ ]:
# Arranque: localizar la raiz del repo y la base de datos DuckDB
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd

from config import config

DB_PATH = ROOT / config.duckdb_path
DB_OK = DB_PATH.exists()
print("BD:", DB_PATH, "->", "OK" if DB_OK else "NO existe (corre `make pipeline`)")


def con():
    if not DB_OK:
        raise FileNotFoundError(f"No existe {DB_PATH}. Ejecuta el pipeline primero.")
    c = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        c.execute("INSTALL spatial; LOAD spatial;")
    except Exception:
        pass
    return c


def q(sql: str) -> pd.DataFrame:
    with con() as c:
        return c.execute(sql).df()


In [ ]:
# Librerias de visualizacion (matplotlib obligatoria, seaborn opcional)
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="viridis")
    HAS_SNS = True
except Exception:
    HAS_SNS = False

plt.rcParams["figure.figsize"] = (10, 5)
NIVEL_COLOR = {"Bajo": "#2e7d32", "Medio": "#f9a825", "Alto": "#ef6c00", "Crítico": "#c62828"}
print("matplotlib listo | seaborn:", HAS_SNS)


In [ ]:
# Nombres desde municipios_geom (no estan en ira_resultados)
ira = q("""
    SELECT r.*, g.nombre_municipio, g.nombre_departamento
    FROM ira_resultados r
    LEFT JOIN municipios_geom g USING (codigo_municipio)
""")
print("ira_resultados:", ira.shape)
ira[["spc", "sep", "sve", "ira_score", "anomaly_score",
     "rendimiento_predicho"]].describe().T


## 1. IRA por nivel de riesgo


In [ ]:
por_nivel = (ira.groupby("ira_nivel")
             .agg(registros=("ira_score", "size"),
                  ira_medio=("ira_score", "mean"),
                  spc=("spc", "mean"), sep=("sep", "mean"), sve=("sve", "mean"))
             .reindex(["Bajo", "Medio", "Alto", "Crítico"]).dropna().round(3))
por_nivel


## 2. Departamentos con mayor riesgo medio


In [ ]:
por_depto = (ira.groupby("nombre_departamento")
             .agg(ira_medio=("ira_score", "mean"), registros=("ira_score", "size"))
             .query("registros >= 20")
             .sort_values("ira_medio", ascending=False).head(15).round(3))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(por_depto.index[::-1], por_depto["ira_medio"][::-1], color="#ef6c00", edgecolor="white")
ax.set_title("Top 15 departamentos por IRA medio (>=20 registros)")
ax.set_xlabel("IRA medio")
plt.tight_layout(); plt.show()
por_depto


## 3. Perfil de riesgo por cultivo


In [ ]:
por_cultivo = (ira.groupby("cultivo")
               .agg(ira_medio=("ira_score", "mean"), registros=("ira_score", "size"))
               .query("registros >= 20")
               .sort_values("ira_medio", ascending=False).head(15).round(3))
por_cultivo


## 4. Matriz de correlaciones

Usamos el dataset normalizado del notebook 02 (o lo recalculamos). Buscamos qué variables
covarían y cuáles empujan el IRA.


In [ ]:
proc = ROOT / "notebooks" / "_data_procesada.parquet"
if proc.exists():
    feats = pd.read_parquet(proc)
else:
    from src.features.normalize import normalize
    feats = normalize(q("SELECT * FROM features_municipio_cultivo"))

# Unimos el IRA score para correlacionar features vs riesgo
base = q("SELECT codigo_municipio, cultivo, periodo, ira_score, spc, sep, sve FROM ira_resultados")
merged = feats.merge(base, on=["codigo_municipio", "cultivo", "periodo"], how="inner")
print("Filas para correlacion:", len(merged))

num = merged.select_dtypes(include=[np.number]).drop(columns=["codigo_municipio"], errors="ignore")
corr = num.corr()
corr.shape


In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
if HAS_SNS:
    sns.heatmap(corr, cmap="RdBu_r", center=0, square=True,
                cbar_kws={"shrink": 0.6}, ax=ax)
else:
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
    fig.colorbar(im, shrink=0.6)
ax.set_title("Matriz de correlaciones — features + sub-indices + IRA")
plt.tight_layout(); plt.show()


## 5. ¿Qué variables se asocian más con el IRA?


In [ ]:
if "ira_score" in corr.columns:
    top = (corr["ira_score"].drop(labels=["ira_score"], errors="ignore")
           .sort_values(key=abs, ascending=False).head(15).round(3))
    print("Correlacion con ira_score (|r| descendente):")
    display(top.to_frame("r"))


In [ ]:
# Aporte de cada sub-indice al IRA (deben dominar segun pesos 0.5 / 0.3 / 0.2)
for s in ["spc", "sep", "sve"]:
    if s in merged.columns:
        r = merged[[s, "ira_score"]].corr().iloc[0, 1]
        print(f"corr({s}, ira_score) = {r:.3f}")


---
### Resumen

- El IRA se concentra en Bajo/Medio; los niveles Alto/Crítico son minoritarios pero son el foco de la alerta.
- La correlación del IRA con cada sub-índice refleja los pesos del modelo (SPC 0.5 > SEP 0.3 > SVE 0.2).
- La matriz de correlaciones identifica variables redundantes y las que más mueven el riesgo.

Siguiente paso → **`04_modelo_predictivo.ipynb`**.
